### Output Chunk Model (RAG-Ready) 

In [3]:
from pydantic import BaseModel
from typing import Any, Dict
from uuid import uuid4

class DocumentChunk(BaseModel):
    id: str
    text: str
    metadata: Dict[str, Any]

    @staticmethod
    def create(text: str, metadata: Dict[str, Any]) -> "DocumentChunk":
        return DocumentChunk(id=str(uuid4()), text=text, metadata=metadata)


### Docling Converter (OCR + Images Enabled)

In [4]:
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    OcrAutoOptions,
)


def build_pipeline_options() -> ThreadedPdfPipelineOptions:
    return ThreadedPdfPipelineOptions(
        do_ocr=True,          # OCR when text is missing
        ocr_options=OcrAutoOptions(lang=["eng"]),
        generate_page_images=True,
        generate_picture_images=True,
    )


### Document Converter

In [5]:
from docling.document_converter import DocumentConverter


def build_converter() -> DocumentConverter:
    return DocumentConverter()


### Document Chunker

In [6]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
import tiktoken


def build_hybrid_chunker_for_ollama(
    max_tokens: int = 768,
) -> HybridChunker:
    # Use a tokenizer encoding compatible with ollama's nomic-embed-text
    try:
        # Prefer nomic-embed-text encoding when available
        encoding = tiktoken.encoding_for_model("nomic-embed-text")
    except Exception:
        # Fallback to a generic encoding compatible with many embedding models
        encoding = tiktoken.get_encoding("cl100k_base")

    tokenizer = OpenAITokenizer(tokenizer=encoding, max_tokens=max_tokens)

    return HybridChunker(
        tokenizer=tokenizer,
        merge_peers=True,   # strongly recommended for RAG
    )


### Image Persistence Helper

In [7]:
from pathlib import Path
from PIL import Image


def save_image(
    image: Image.Image,
    source_file: str,
    page_number: int,
    image_index: int,
    output_dir: str = "images",
) -> str:
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    image_path = (
        Path(output_dir)
        / f"{source_file}_p{page_number}_img{image_index}.png"
    )

    image.save(image_path)
    return str(image_path)


Convert + Chunk + Normalize (Core Pipeline)

In [8]:
from typing import List
from pathlib import Path
from docling.datamodel.base_models import InputFormat


def _extract_meta_fields(chunk):
    meta_dict = chunk.meta.export_json_dict() if hasattr(chunk.meta, "export_json_dict") else {}

    # page numbers
    doc_items = meta_dict.get("doc_items", []) or []
    page_numbers = []
    for di in doc_items:
        if isinstance(di, dict):
            page = di.get("page") or di.get("page_number") or di.get("page_no")
            if page is not None:
                page_numbers.append(page)
    page_numbers = sorted(set(page_numbers)) if page_numbers else None

    # bbox (try first doc_item)
    bbox = None
    if doc_items and isinstance(doc_items[0], dict):
        bbox = doc_items[0].get("bbox")

    # other flags
    has_ocr = meta_dict.get("has_ocr", getattr(chunk, "has_ocr", False))
    language = meta_dict.get("language", getattr(chunk, "language", None))

    return page_numbers, bbox, has_ocr, language


def process_document_with_ollama(
    file_path: str,
    max_tokens: int = 768,
) -> List[DocumentChunk]:
    source_file = Path(file_path).name

    # ---- Build components ----
    converter = build_converter()
    pipeline_options = build_pipeline_options()
    # Attach pipeline options to PDF format so DocumentConverter uses them
    converter.format_to_options[InputFormat.PDF].pipeline_options = pipeline_options

    chunker = build_hybrid_chunker_for_ollama(max_tokens=max_tokens)

    # ---- Convert document (OCR + parsing) ----
    conversion_result = converter.convert(
        file_path,
    )

    document = conversion_result.document

    # ---- Hybrid chunking ----
    docling_chunks = list(chunker.chunk(dl_doc=document))

    output_chunks: List[DocumentChunk] = []
    image_counter = 0

    for chunk in docling_chunks:
        page_numbers, bbox, has_ocr, language = _extract_meta_fields(chunk)

        # ---- TEXT / TABLE / CAPTION ----
        if getattr(chunk, "text", None) and chunk.text.strip():
            output_chunks.append(
                DocumentChunk.create(
                    text=chunk.text,
                    metadata={
                        "source_file": source_file,
                        "page_numbers": page_numbers,
                        "chunk_type": None,
                        "bbox": bbox,
                        "has_ocr": has_ocr,
                        "language": language,
                        "image_path": None,
                    },
                )
            )

        # ---- IMAGE ----
        img_obj = getattr(chunk, "image", None)
        # If underlying doc_item contains an image ref, it may be available via meta export as a dict
        if img_obj is None:
            meta_dict = chunk.meta.export_json_dict() if hasattr(chunk.meta, "export_json_dict") else {}
            doc_items = meta_dict.get("doc_items", []) or []
            if doc_items and isinstance(doc_items[0], dict):
                img_ref = doc_items[0].get("image_ref")
                if img_ref:
                    # image_ref.uri might be a file path or data url; skip complex handling for now
                    img_obj = None

        if img_obj is not None:
            page_number = page_numbers[0] if page_numbers else -1

            image_path = save_image(
                image=img_obj,
                source_file=source_file,
                page_number=page_number,
                image_index=image_counter,
            )

            image_counter += 1

            output_chunks.append(
                DocumentChunk.create(
                    text="",
                    metadata={
                        "source_file": source_file,
                        "page_numbers": page_numbers,
                        "chunk_type": "image",
                        "bbox": bbox,
                        "has_ocr": False,
                        "language": None,
                        "image_path": image_path,
                    },
                )
            )

    return output_chunks


### Example Usage

In [9]:
import os
from IPython.display import JSON, display

source = os.path.join(os.getcwd(), "1984.pdf")

chunks = process_document_with_ollama(file_path=source, max_tokens=768)



2025-12-28 18:51:40,535 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-28 18:51:40,565 - INFO - Going to convert document batch...
2025-12-28 18:51:40,566 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 172b3de10c9ed12a7c2a9458e55cf3e3
2025-12-28 18:51:40,579 - INFO - Loading plugin 'docling_defaults'
2025-12-28 18:51:40,582 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-28 18:51:40,596 - INFO - Loading plugin 'docling_defaults'
2025-12-28 18:51:40,602 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-28 18:51:40,634 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-12-28 18:51:40,723 - INFO - Accelerator device: 'cpu'
2025-12-28 18:51:42,306 - INFO - Auto OCR model selected easyocr.
2025-12-28 18:51:42,320 - INFO - Loading plugin 'docling_defaults'
2025-12-28 18:51:42,325 - INFO - Registered layout engines: ['docling_layout_default', 'docling_e

### Display Processing Output

In [13]:
data = chunks[3].model_dump()

display(JSON({"id": data.get("id")}))
display(JSON({"text": data.get("text")}))
display(JSON({"metadata": data.get("metadata")}))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>